# DBAD 4000 — Week 11 · Build TrailPeak's lakehouse
## Files you can trust, change, rewind and stream into

Last week you queried TrailPeak's 254,366 storefront events as **files**. Today you turn those files into
**tables** — Delta Lake tables — and find out what a table format adds: a history you can query, changes you can
undo, columns you can add without breaking yesterday's queries, a pipeline from raw to report, and a stream that
picks up new files as they arrive.

**How to use it.** Run the cells top to bottom with the instructor. Every code cell prints something; a
**Check** line under it says what you should see. Two *Your turn* stops ask you to write the query yourself.

**Where to run it (browser only — nothing installs on your laptop):** Google Colab is the default. Databricks
Free Edition is the alternative: import `wk11_lakehouse_databricks.ipynb` instead — the same cells with the
Databricks paths set and the two Colab-only cells replaced by notes.

**AI-use policy:** AI tools are allowed for brainstorming and syntax **with attribution**. Unattributed or
unmodified AI-generated work is an academic-integrity violation (SAIT AC.3.4.3, AD.2.15.3).

## Setup — two cells

1. Download **`wk11_lake_bundle.zip`** (about 4 MB) from D2L → Week 11.
2. **Colab:** left sidebar → **Files** (folder icon) → **Upload to session storage** → the zip. *Uploads disappear
   when the runtime ends* — after a long break, upload again (or keep the zip in Google Drive, click **Mount Drive**
   and set `ZIP_PATH` to `/content/drive/MyDrive/wk11_lake_bundle.zip`).
3. Run the install cell, then the loader. **Check:** `Spark 4.0.5 ready · Delta 4.0.1` (your Spark version may
   differ) and `lake: 254,366 events in 36 files`.

In [ ]:
# Colab only, once per session (about 20 s): Delta Lake for the PySpark that Colab already has (4.0.x).
# Databricks: this cell is not in your copy; Delta is built in.
%pip install -q delta-spark==4.0.1

In [ ]:
# --- wk11 loader (completed for you: read it, it teaches) ------------------------------------------
import os, re, json, shutil, zipfile, collections

ZIP_PATH = "wk11_lake_bundle.zip"     # Drive: "/content/drive/MyDrive/wk11_lake_bundle.zip"
DATA_DIR = "wk11_data"                 # where the zip is unpacked
if os.path.exists(ZIP_PATH) and not os.path.exists(DATA_DIR):
    with zipfile.ZipFile(ZIP_PATH) as z:
        z.extractall(DATA_DIR)
    print(f"unzipped {ZIP_PATH} -> {DATA_DIR}/")

LAKE    = f"{DATA_DIR}/lake/events"   # the lake: Parquet files in year=/month= folders
BATCHES = f"{DATA_DIR}/batches"       # files that arrive later: late, July, CDC
DROPS   = f"{DATA_DIR}/stream"        # three small files for the streaming part
WORK    = f"{DATA_DIR}/work"          # scratch space: a copied folder, the stream's inbox and checkpoint

from pyspark.sql import SparkSession, functions as F
try:
    spark                              # Databricks: a session already exists, with Delta built in
except NameError:                      # Colab: start Spark with the Delta extension switched on
    from delta import configure_spark_with_delta_pip
    shutil.rmtree("spark-warehouse", ignore_errors=True)    # tables left by an earlier session
    builder = (SparkSession.builder.appName("wk11-lakehouse")
               .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
               .config("spark.sql.catalog.spark_catalog",
                       "org.apache.spark.sql.delta.catalog.DeltaCatalog"))
    spark = configure_spark_with_delta_pip(builder).getOrCreate()
spark.conf.set("spark.sql.session.timeZone", "UTC")         # the data is UTC; cut months in UTC
try:
    spark.sparkContext.setLogLevel("ERROR")                  # quieter; Databricks refuses this call
except Exception:
    pass

# the 13 columns of every event, as in Week 10
EVENTS_SCHEMA = """event_id STRING, event_ts TIMESTAMP, event_type STRING, session_id STRING,
    customer_id INT, platform STRING, city STRING, product_id INT, store_id INT,
    quantity INT, unit_price DOUBLE, search_query STRING, page_url STRING"""
from importlib.metadata import version, PackageNotFoundError
try:
    delta_version = version("delta-spark")
except PackageNotFoundError:
    delta_version = "built in"
print("Spark", spark.version, "ready · Delta", delta_version)
print(f"lake: {spark.read.parquet(LAKE).count():,} events in "
      f"{sum(len(f) for _, _, f in os.walk(LAKE))} files")

## Part A · The lake is a folder of files

A **data lake** keeps data as files in cheap storage, in whatever format it arrived. TrailPeak's lake holds the
254,366 events as Parquet, one file per month, in folders named `year=…/month=…` (Hive style: the folder
name *is* the value). Nothing here is a table yet: there is no catalog entry, no history and no rule about what a
writer may add.

In [ ]:
print(sorted(os.listdir(LAKE)))                  # one folder per year ...
print(sorted(os.listdir(f"{LAKE}/year=2026")))   # ... and one per month inside it

**Check:** four `year=` folders, and `month=01` … `month=06` for 2026 (the data runs to June 2026).

In [ ]:
lake = spark.read.parquet(LAKE)                  # schema-on-read: Spark finds the columns now
print(f"{lake.count():,} rows from {len(lake.inputFiles())} files")
print(lake.columns[-4:])                         # year and month came from the folder names

**Check:** `254,366 rows from 36 files`, and the last columns are `search_query`, `page_url`, `year`, `month`.

In [ ]:
june_sql = f"""
SELECT count(*) AS events,
       sum(CASE WHEN event_type = 'purchase' THEN 1 ELSE 0 END) AS purchase_lines
FROM parquet.`{LAKE}`
WHERE year = 2026 AND month = 6
"""
spark.sql(june_sql).show()                       # SQL straight on the files: nothing loaded first
plan = spark.sql("EXPLAIN " + june_sql).first()[0]
print(re.search(r"PartitionFilters: \[[^\]]*\]", plan).group(0))   # only one folder is read

**Check:** `11,981` events and `431` purchase lines; the plan's `PartitionFilters` names `year = 2026` and `month = 6`, so Spark opens one month folder of 36.

In [ ]:
swamp = f"{WORK}/swamp_june"                    # June 2026 copied out as a plain folder of Parquet files
(lake.filter("year = 2026 AND month = 6").drop("year", "month")
     .write.mode("overwrite").parquet(swamp))
late = spark.read.parquet(f"{BATCHES}/late_events_2026-06.parquet")
late.write.mode("append").parquet(swamp)         # tonight's job adds the late events ...
late.write.mode("append").parquet(swamp)         # ... and someone re-runs it by mistake

(spark.read.parquet(swamp)
      .agg(F.count("*").alias("rows"), F.countDistinct("event_id").alias("distinct_ids"))
      .show())
print(sorted(f for f in os.listdir(swamp) if f.endswith(".parquet")))   # which file is the duplicate?

**Check:** `rows` 12,895 but `distinct_ids` 12,438: 457 events are in the folder
twice. The file names are random, nothing recorded which write made which file, and there is no undo — you would
have to find and delete files by hand while nobody else reads the folder. That is the problem a table format solves.

## Part B · From files to a table: Delta Lake

**Delta Lake** is an *open table format*: the data stays in ordinary Parquet files, and a folder called
`_delta_log` beside them records every change as a numbered **commit** (a small JSON file). The table at
**version** *n* is whatever commits 0 to *n* say. Readers see only finished commits, so a half-written job is
never visible, and two writers cannot both win — the ACID guarantees from Week 5, on plain files.

In [ ]:
spark.sql(f"""
CREATE OR REPLACE TABLE bronze_events USING DELTA AS
SELECT event_id, event_ts, event_type, session_id, customer_id, platform, city,
       product_id, store_id, quantity, unit_price, search_query, page_url
FROM parquet.`{LAKE}`
""")
spark.sql("SELECT count(*) AS events FROM bronze_events").show()

**Check:** `254366`. The same rows, now a table called `bronze_events` (bronze = the raw data, as it landed). `USING DELTA` makes it a Delta table; on Databricks it is the default.

In [ ]:
(spark.sql("DESCRIBE HISTORY bronze_events")
      .select("version", "operation",
              F.col("operationMetrics")["numOutputRows"].alias("rows_written"),
              F.col("operationMetrics")["numFiles"].alias("files"))
      .show(truncate=False))

**Check:** one row: version `0`, `CREATE OR REPLACE TABLE AS SELECT`, `254366` rows written. This history *is* the transaction log, shown as a table.

In [ ]:
# Colab only: look inside the table's folder. (Databricks hides managed tables' files; use DESCRIBE HISTORY.)
loc = spark.sql("DESCRIBE DETAIL bronze_events").first()["location"].removeprefix("file:")
log = f"{loc}/_delta_log"
print(sorted(f for f in os.listdir(log) if f.endswith(".json")))
with open(f"{log}/00000000000000000000.json") as fh:
    actions = [next(iter(json.loads(line))) for line in fh]   # each line is one action
print(collections.Counter(actions))

**Check:** one commit file, `00000000000000000000.json`, holding `commitInfo`, `protocol`, `metaData` (the
schema) and one `add` action per Parquet file written. Each later change appends the next numbered file.

## Part C · Change it, and rewind it

Now the same late batch that broke the plain folder goes into the table — twice, by mistake — and the table's
history undoes it. Every statement below writes a new version; `VERSION AS OF n` reads an old one (**time travel**).

In [ ]:
spark.sql(f"INSERT INTO bronze_events SELECT * FROM parquet.`{BATCHES}/late_events_2026-06.parquet`")
spark.sql("SELECT count(*) AS events FROM bronze_events").show()

**Check:** `254823` = 254,366 + 457 late events. That INSERT was version 1.

In [ ]:
JUNE = """
SELECT count(*) AS purchase_lines, round(sum(quantity * unit_price), 2) AS revenue
FROM bronze_events {v}
WHERE event_type = 'purchase'
  AND event_ts >= '2026-06-01' AND event_ts < '2026-07-01'
"""
spark.sql(JUNE.format(v="")).show()                   # June now
spark.sql(JUNE.format(v="VERSION AS OF 0")).show()    # June as the table was before the late batch

**Check:** now 462 lines, $174,914.92; at version 0, 431 lines, $164,370.48. The report you sent on 1 July can be reproduced exactly, after the data changed.

In [ ]:
# Oops: the same INSERT runs a second time
spark.sql(f"INSERT INTO bronze_events SELECT * FROM parquet.`{BATCHES}/late_events_2026-06.parquet`")
spark.sql("SELECT count(*) - count(DISTINCT event_id) AS duplicates FROM bronze_events").show()

**Check:** `457` duplicates — the same mistake as Part A. This time the table knows exactly which commit did it.

In [ ]:
(spark.sql("DESCRIBE HISTORY bronze_events")
      .select("version", "operation", F.col("operationMetrics")["numOutputRows"].alias("rows_written"))
      .orderBy("version").show())

**Check:** versions 0, 1 and 2: the create, the INSERT and the accidental second INSERT, each with its row count (versions 1 and 2 show `WRITE`, 457 rows).

In [ ]:
spark.sql("RESTORE TABLE bronze_events TO VERSION AS OF 1").show()
spark.sql("SELECT count(*) - count(DISTINCT event_id) AS duplicates FROM bronze_events").show()

**Check:** the restore reports the files it removed and re-added, and `duplicates` is `0`. RESTORE did not erase version 2: it wrote version 3, whose contents equal version 1. History only grows.

In [ ]:
FORGET = 48          # this customer asked TrailPeak to delete their data
spark.sql(f"DELETE FROM bronze_events WHERE customer_id = {FORGET}").show()
spark.sql(f"SELECT count(*) AS rows_now FROM bronze_events WHERE customer_id = {FORGET}").show()
spark.sql(f"""SELECT count(*) AS rows_at_v3 FROM bronze_events VERSION AS OF 3
              WHERE customer_id = {FORGET}""").show()

**Check:** `num_affected_rows` 39, `rows_now` 0, `rows_at_v3` 39. The DELETE is version 4, and
a plain folder could not have done it without rewriting files by hand. But time travel still finds the rows: old
files stay until **VACUUM** removes them (by default only files older than 7 days). Deleting personal data means
DELETE *and* VACUUM — Week 14 returns to this.

### Your turn 1 — read the table's history

Write each answer in the empty cell under it. Use `VERSION AS OF` and `DESCRIBE HISTORY`.

**1a.** How many events did the late batch add? Compare the row counts at version 0 and version 1. *Check: 457.*

**1b.** Which platform sent the most late events? List version 1's events that are not in version 0 (hint:
`WHERE event_id NOT IN (SELECT event_id FROM bronze_events VERSION AS OF 0)`), grouped by `platform`.
*Check: ios 259, android 198.*

**1c.** In `DESCRIBE HISTORY bronze_events`, which version deleted rows, and how many? (Look in
`operationMetrics` for `numDeletedRows`.) *Check: version 4, 39.*

In [ ]:
# 1a: your query here

In [ ]:
# 1b: your query here

In [ ]:
# 1c: your query here

## Part D · A new column arrives: schema enforcement, then schema evolution

On 1 July 2026 TrailPeak launched promo codes, and the July batch (JSON lines this time) carries a 14th column,
`promo_code`. A Delta table checks every write against its schema (**schema enforcement**) and refuses a mismatch
— unless you say the schema may grow (**schema evolution**).

In [ ]:
JULY_SCHEMA = EVENTS_SCHEMA + ", promo_code STRING"           # the 13 columns plus the new one
july = spark.read.schema(JULY_SCHEMA).json(f"{BATCHES}/july_2026_promo.json")
print(f"{july.count():,} events, {len(july.columns)} columns "
      f"(the table has {len(spark.table('bronze_events').columns)})")
july.filter("promo_code IS NOT NULL").select("event_id", "event_ts", "unit_price", "promo_code").show(3)

**Check:** `2,889 events, 14 columns (the table has 13)`, and three purchase lines with a code.

In [ ]:
try:
    july.write.format("delta").mode("append").saveAsTable("bronze_events")
except Exception as e:                                         # the table refuses the write
    print(type(e).__name__, "-", str(e).splitlines()[0][:110])

**Check:** an `AnalysisException` saying a schema mismatch was detected. Nothing was written: the table is still at version 4.

In [ ]:
(july.write.format("delta").mode("append")
     .option("mergeSchema", "true")                            # allow the schema to grow
     .saveAsTable("bronze_events"))
spark.sql("""SELECT promo_code, count(*) AS events
             FROM bronze_events GROUP BY promo_code ORDER BY events DESC""").show()

**Check:** `NULL` 257,624 (every older event, which never had a code), `SUMMIT10` 39, `TRAIL15` 10. Version 5.

In [ ]:
spark.sql(JUNE.format(v="")).show()         # yesterday's June query, unchanged

**Check:** 462 lines, $174,914.92 — the same answer as before the new column: adding a column broke no existing query. (`ALTER TABLE bronze_events ADD COLUMNS (…)` does the same in SQL.)

## Part E · A pipeline: bronze → silver → gold, then change data capture

The **medallion** layout is how most lakehouses organize tables:

* **bronze** — raw, as it landed (`bronze_events`);
* **silver** — cleaned and typed: one row per purchase line, money as `DECIMAL`, test traffic removed;
* **gold** — business-level answers that dashboards read: revenue per store per month.

Loading raw data first and transforming it inside the engine with SQL is **ELT** (extract, load, transform).

In [ ]:
spark.sql("""
CREATE OR REPLACE TABLE silver_purchases USING DELTA AS
SELECT event_id, event_ts, customer_id, store_id, product_id, platform, promo_code,
       quantity,
       CAST(unit_price AS DECIMAL(10,2))            AS unit_price,
       quantity * CAST(unit_price AS DECIMAL(10,2)) AS line_total
FROM bronze_events
WHERE event_type = 'purchase'
  AND session_id NOT LIKE 's-TEST%'                 -- drop the QA team's test session
""")
spark.sql("""SELECT count(*) AS lines, count(DISTINCT event_id) AS ids, sum(line_total) AS revenue
             FROM silver_purchases""").show()

**Check:** 9,320 lines, 9,320 distinct ids (one row per purchase line), revenue 3223964.53. The 2 penny purchases from the test session are gone.

In [ ]:
def build_gold():
    spark.sql("""
    CREATE OR REPLACE TABLE gold_store_month USING DELTA AS
    SELECT store_id,
           date_format(event_ts, 'yyyy-MM') AS month,
           count(*)                         AS lines,
           sum(line_total)                  AS revenue
    FROM silver_purchases
    GROUP BY store_id, date_format(event_ts, 'yyyy-MM')
    """)

build_gold()
spark.sql("SELECT * FROM gold_store_month WHERE month = '2026-06' ORDER BY store_id").show()

**Check:** four rows for June 2026; store 1 has 158 lines and 66446.61. The whole gold table has 148 rows: 4 stores × 37 months.

In [ ]:
CDC_SCHEMA = """op STRING, event_id STRING, event_ts TIMESTAMP, customer_id INT, store_id INT,
    product_id INT, platform STRING, quantity INT, unit_price DECIMAL(10,2), changed_at TIMESTAMP"""
changes = spark.read.csv(f"{BATCHES}/cdc_purchases.csv", header=True, schema=CDC_SCHEMA)
changes.createOrReplaceTempView("changes")       # a name SQL can use, for this session
spark.sql("SELECT op, count(*) AS changes FROM changes GROUP BY op ORDER BY op").show()

**Check:** `D` 12 (cancelled lines), `I` 8 (phone orders the storefront never logged), `U` 20 (partial returns and price matches). This file is the order system's change feed: **CDC**, change data capture.

In [ ]:
spark.sql("""
MERGE INTO silver_purchases AS t
USING changes AS c
ON t.event_id = c.event_id
WHEN MATCHED AND c.op = 'D' THEN DELETE
WHEN MATCHED AND c.op = 'U' THEN UPDATE SET
     quantity = c.quantity, unit_price = c.unit_price, line_total = c.quantity * c.unit_price
WHEN NOT MATCHED AND c.op = 'I' THEN INSERT
     (event_id, event_ts, customer_id, store_id, product_id, platform, promo_code,
      quantity, unit_price, line_total)
     VALUES (c.event_id, c.event_ts, c.customer_id, c.store_id, c.product_id, c.platform, NULL,
             c.quantity, c.unit_price, c.quantity * c.unit_price)
""").show()

**Check:** `num_affected_rows` 40 = 20 updated + 12 deleted + 8 inserted, in one commit. MERGE is an *upsert*: update the rows that match, insert the ones that do not, delete where told to.

In [ ]:
build_gold()                                      # re-run the gold step: version 1 of gold_store_month
spark.sql("""
SELECT n.store_id, o.revenue AS before, n.revenue AS after, n.revenue - o.revenue AS change
FROM gold_store_month n
JOIN gold_store_month VERSION AS OF 0 o ON n.store_id = o.store_id AND n.month = o.month
WHERE n.month = '2026-06'
ORDER BY n.store_id
""").show()

**Check:** store 1 -710.17, store 2 -937.12, store 3 +795.70, store 4 +1341.40: cancellations and returns pull revenue down; the phone orders push stores 3 and 4 up.

### Your turn 2 — what did the change feed change?

**2a.** By how much did the MERGE change total revenue in `silver_purchases`? Compare `sum(line_total)` now with
`VERSION AS OF 0`. *Check: -2157.41.*

**2b.** How many phone orders are in silver now, and what is their revenue? (`platform = 'phone'`)
*Check: 8 lines, 4919.84.*

**2c.** In `DESCRIBE HISTORY silver_purchases`, read the MERGE's `numTargetRowsUpdated`, `numTargetRowsDeleted`
and `numTargetRowsInserted`. *Check: 20, 12, 8.*

In [ ]:
# 2a: your query here

In [ ]:
# 2b: your query here

In [ ]:
# 2c: your query here

## Part F · Streaming: new files, picked up as they arrive

A **stream** is a query that keeps going: each time it runs it processes only data it has not seen, and remembers
where it stopped. Spark's **Structured Streaming** works in **micro-batches**. Here the storefront drops its event
files into an inbox folder during 8 July 2026; in production the source would be a Kafka topic, and only the first
line of the reading code would change.

In [ ]:
INBOX      = f"{WORK}/inbox"                         # where new files land
CHECKPOINT = f"{WORK}/checkpoints/bronze_live"       # where the stream remembers what it has done
spark.sql("DROP TABLE IF EXISTS bronze_live")          # start clean if you re-run this part
shutil.rmtree(INBOX, ignore_errors=True)
shutil.rmtree(CHECKPOINT, ignore_errors=True)
os.makedirs(INBOX)

def deliver(n):
    """Pretend the storefront has just dropped its next file into the inbox."""
    shutil.copy(f"{DROPS}/drop_{n:02d}.json", INBOX)
    print("inbox:", sorted(os.listdir(INBOX)))

deliver(1)

In [ ]:
def run_stream():
    live = spark.readStream.schema(JULY_SCHEMA).json(INBOX)    # a stream over the inbox folder
    q = (live.writeStream.format("delta")
             .option("checkpointLocation", CHECKPOINT)          # files already done are skipped
             .trigger(availableNow=True)                        # take what has arrived, then stop
             .toTable("bronze_live"))
    q.awaitTermination()
    print("rows in this run:", sum(p["numInputRows"] for p in q.recentProgress))
    spark.sql("SELECT count(*) AS rows_in_table FROM bronze_live").show()

run_stream()

**Check:** `rows in this run: 90` and `rows_in_table` 90: drop 1, 8 July before 10:00.

In [ ]:
deliver(2)
deliver(3)
run_stream()                 # only the two new files are read

**Check:** `rows in this run: 347` (116 + 231) and `rows_in_table` 437. Drop 1 was not read again: the checkpoint remembered it.

In [ ]:
run_stream()                 # nothing new has arrived

**Check:** `rows in this run: 0` and still 437 rows. Run it twice, get each event once.

In [ ]:
spark.sql("""SELECT event_type, count(*) AS events FROM bronze_live
             GROUP BY event_type ORDER BY events DESC""").show()

**Check:** `product_view` 210 at the top, `purchase` 13. A dashboard re-reads this table; each run of the stream adds what is new.

**The same stream from Kafka** (shown, not run — there is no Kafka server here). Only the source changes; the
`writeStream`, the checkpoint and the table stay exactly as above:

```python
live = (spark.readStream.format("kafka")
        .option("kafka.bootstrap.servers", "broker1:9092")
        .option("subscribe", "trailpeak.events")
        .load()
        .select(F.from_json(F.col("value").cast("string"), JULY_SCHEMA).alias("e"))
        .select("e.*"))
```

## What you built today

| Step | Table | Version history |
|---|---|---|
| The lake, read in place | (files only) | none |
| Raw events | `bronze_events` | 0 create · 1 late batch · 2 oops · 3 restore · 4 delete · 5 July with `promo_code` |
| Clean purchase lines | `silver_purchases` | 0 create · 1 MERGE of the change feed |
| Revenue per store per month | `gold_store_month` | 0 build · 1 rebuild after the MERGE |
| Live events | `bronze_live` | one version per stream run that found new files |

The lab continues in this notebook's tables. Keep the runtime connected, or re-run all cells before starting it.

## After class (optional)

Each cell below runs on its own after the cells above. Answers are not given; the output is the check.

In [ ]:
# Time travel by time instead of number: pick a timestamp from DESCRIBE HISTORY, then
ts = spark.sql("DESCRIBE HISTORY bronze_events").filter("version = 1").first()["timestamp"]
spark.sql(f"SELECT count(*) AS events FROM bronze_events TIMESTAMP AS OF '{ts}'").show()

In [ ]:
# Colab only. VACUUM really erases: shorten the retention (never do this on a shared table), then try time travel.
spark.conf.set("spark.databricks.delta.retentionDurationCheck.enabled", "false")
spark.sql("VACUUM bronze_events RETAIN 0 HOURS").show()
try:
    spark.sql("SELECT count(*) FROM bronze_events VERSION AS OF 3").show()
except Exception as e:
    print(type(e).__name__, "-", str(e).splitlines()[0][:120])

In [ ]:
# Change Data Feed: let a table publish its own row-level changes (CDC going out instead of coming in)
spark.sql("ALTER TABLE gold_store_month SET TBLPROPERTIES (delta.enableChangeDataFeed = true)")
v = spark.sql("DESCRIBE HISTORY gold_store_month").agg(F.max("version")).first()[0]
spark.sql("UPDATE gold_store_month SET revenue = revenue + 1 WHERE store_id = 4 AND month = '2026-06'")
spark.sql(f"SELECT store_id, month, revenue, _change_type FROM table_changes('gold_store_month', {v + 1})").show()

In [ ]:
# The same ideas in Apache Iceberg, with no Spark at all: PyIceberg and a local SQLite catalog (about 30 s to install)
%pip install -q "pyiceberg[sql-sqlite,pyarrow]"

In [ ]:
import pyarrow.parquet as pq
from pyiceberg.catalog.sql import SqlCatalog
os.makedirs(f"{WORK}/iceberg", exist_ok=True)
cat = SqlCatalog("local", uri=f"sqlite:///{WORK}/iceberg/catalog.db", warehouse=f"file://{os.path.abspath(WORK)}/iceberg")
cat.create_namespace_if_not_exists("trailpeak")
june = pq.read_table(f"{LAKE}/year=2026/month=06/part-00000.parquet")
tbl = cat.create_table_if_not_exists("trailpeak.june_events", schema=june.schema)
tbl.append(june)                                                        # snapshot 1
tbl.append(pq.read_table(f"{BATCHES}/late_events_2026-06.parquet"))  # snapshot 2
for s in tbl.snapshots():
    print(s.snapshot_id, s.summary["operation"], "total-records =", s.summary["total-records"])
first = tbl.snapshots()[0].snapshot_id
print("rows at the first snapshot:", tbl.scan(snapshot_id=first).to_arrow().num_rows)